# 03 - Bright/faint split + baseline model

Splits on `mag_F160W` (this catalog's reference band, standing in for SDSS's "r") at its median: **bright** -> train + calibration (stands in for a spectroscopic sample), **faint** -> target, held out and never trained on. The reference magnitude is kept as a model feature, so the selection depends only on the inputs -- covariate shift holds by construction.

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb

df = pd.read_parquet("../data/processed/features.parquet")
FEATURE_COLS = json.load(open("../data/processed/feature_cols.json"))

In [2]:
threshold = df["mag_F160W"].median()
bright = df[df["mag_F160W"] <= threshold].copy()
faint = df[df["mag_F160W"] > threshold].copy()
print(f"threshold={threshold:.2f}  bright n={len(bright)} (z_spec median {bright.z_spec.median():.2f})"
      f"  faint n={len(faint)} (z_spec median {faint.z_spec.median():.2f})")

threshold=21.57  bright n=1986 (z_spec median 0.67)  faint n=1985 (z_spec median 1.02)


In [3]:
# train/calibration split of the bright set only -- faint set is the held-out target
train_df, cal_df = train_test_split(bright, test_size=0.3, random_state=42)
len(train_df), len(cal_df), len(faint)

(1390, 596, 1985)

In [4]:
model = xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
model.fit(train_df[FEATURE_COLS], train_df["z_spec"])

cal_df["z_pred"] = model.predict(cal_df[FEATURE_COLS])
faint["z_pred"] = model.predict(faint[FEATURE_COLS])

## Point-prediction sanity check (not the headline result -- that's coverage, next notebooks)

In [5]:
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

print("calibration (bright, held out) RMSE:", rmse(cal_df["z_spec"], cal_df["z_pred"]))
print("target (faint)               RMSE:", rmse(faint["z_spec"], faint["z_pred"]))

calibration (bright, held out) RMSE: 0.24071457860574139
target (faint)               RMSE: 0.6994425831752463


In [6]:
cal_df.to_parquet("../data/processed/cal_with_preds.parquet", index=False)
faint.to_parquet("../data/processed/target_with_preds.parquet", index=False)
json.dump({"threshold": float(threshold)}, open("../data/processed/split_meta.json", "w"))